# Success-only collection in narrow square bands across the right room

Use a GPU runtime. The six collection goals and fifteen withheld goals are the same as in the earlier half-map radial run. Starts are random within each newly added square strip.

- **Eleven stages.** Half-widths are **6, 10, 14, 18, 22, 26, 30, 34, 38, 42, and 48**. Most strips are 4 units wide. The final extent spans the whole right room around each training goal. Starts are sampled within the simulator's reset-safe area. A goal with an empty strip is skipped for that band, giving **342 default rollout attempts**.
- **Success-only labels.** Band 1 uses raw MSE. Each later band collects with the previous checkpoint frozen. Among eligible consecutive frames in successful rollouts, time order alone provides the label. Retain up to eight pairs per rollout. Failed rollouts provide zero pairs. Replay can come from anywhere in the right room.
- **Fit cumulatively.** Start from identity and run 500 optimizer updates after any band with new pairs. A band with zero new pairs carries forward the incoming adapter. The report shows how much successful experience each strip actually supplied.
- **Evaluate the entire right half-map.** Compare raw and adapted costs on the full held-out grid. Also report the original restricted evaluation cohort for comparison with the earlier radial run.

Allow several GB of Drive space. To resume, set `RESUME_RUN_ID` and use the same numerical runtime.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
SOURCE_RUN=DRIVE/'datasets/curriculum_cost_adapter/radial_multigoal_ranking_v1/20261106T195304628603Z'
assert (SOURCE_RUN/'branches/multi/complete.json').is_file(), 'Set SOURCE_RUN to your completed multi-goal ranking run.'
RADIAL_RUN=DRIVE/'datasets/curriculum_cost_adapter/experience_temporal_v1/20261107T142845862912Z'
RESUME_RUN_ID=None  # Set to the printed ID to resume an interrupted run.
STARTS_PER_GOAL=6
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/experience_temporal_half_square_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-experience-temporal-half-square')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='548b76e574706be48578b256845a615156e2943e'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/experience-temporal-half-square-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Frozen ranking source:',SOURCE_RUN)
print('Run ID:',RUN_ID)
print('New experience curriculum results:',OUTPUT)

## Load the pinned code and frozen models

The world-model checkpoint, encoder and renderer are the same ones used in the previous planning comparison.

If Drive interrupts a checksum read, the code retries and prints the file path. For a persistent error, remount Drive and rerun the affected band.

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

def command(operation, band=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.experience_curriculum',operation,
          '--source-run',SOURCE_RUN,'--output-dir',OUTPUT,'--world-model-checkpoint',CHECKPOINT,
          '--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda',
          '--starts-per-goal',STARTS_PER_GOAL,'--labels','temporal','--sampling','half-square']
    if band is not None:
        args+=['--band',band]
    return args

def run_band(band):
    started=time.monotonic()
    shell(command('stage',band),cwd=REPO)
    print('Band minutes:',round((time.monotonic()-started)/60,2))

## Band 1 · square half-width 0–6

In [ ]:
run_band(1)

## Band 2 · square half-width 6–10

In [ ]:
run_band(2)

## Band 3 · square half-width 10–14

In [ ]:
run_band(3)

## Band 4 · square half-width 14–18

In [ ]:
run_band(4)

## Band 5 · square half-width 18–22

In [ ]:
run_band(5)

## Band 6 · square half-width 22–26

In [ ]:
run_band(6)

## Band 7 · square half-width 26–30

In [ ]:
run_band(7)

## Band 8 · square half-width 30–34

In [ ]:
run_band(8)

## Band 9 · square half-width 34–38

In [ ]:
run_band(9)

## Band 10 · square half-width 38–42

In [ ]:
run_band(10)

## Band 11 · square half-width 42–48

In [ ]:
run_band(11)

## Compare the learned geometry

The first table uses the entire held-out right-room grid. The second uses the smaller region shared with the radial run.

In [ ]:
shell(command('compare'),cwd=REPO)
report=json.loads((OUTPUT/'comparison.json').read_text())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

metrics=[]
for method, rows in report['methods'].items():
    for goal_id, row in enumerate(rows):
        order=row['adapted_rank']
        steps=row['heldout_directional']
        metrics.append(dict(method=method,goal_id=goal_id,goal=tuple(row['goal']),
                            goal_type='withheld' if goal_id else 'seen',queries=row['heldout_count'],
                            correct_order=order['correct_fraction'],pair_ties=order['tie_fraction'],
                            uphill=steps['adapted_up_fraction'],step_ties=steps['adapted_tie_fraction']))
table=pd.DataFrame(metrics)
columns=['correct_order','pair_ties','uphill','step_ties']
print('Full right room · average per withheld goal (percent)')
display((100*table[table.goal_type=='withheld'].groupby('method',sort=False)[columns].mean()).round(2))
print('Per-goal results (fractions)')
display(table.round(4))
table.to_csv(OUTPUT/'per_goal_metrics.csv',index=False)

collected=pd.DataFrame([
    dict(band=band,goal=tuple(row['case']['goal']),
         rollouts=1,successes=int(row['success']),steps=row['steps'],stationary=row['stationary_steps'],
         failed_skipped=int(row['skipped_failed']),eligible=row['eligible_pairs'],retained=row['retained_pairs'])
    for band,stage in report['checkpoints'].items() for row in stage['collection']
])
print('Collection totals per band')
display(collected.groupby('band').sum(numeric_only=True))
print('Collection totals per goal and band')
display(collected.groupby(['band','goal']).sum(numeric_only=True))
collected.to_csv(OUTPUT/'collection_counts.csv',index=False)
print('Optimizer updates by band')
display(pd.Series({band:s['updates'] for band,s in report['checkpoints'].items()},name='updates'))

def old_cohort(methods, label):
    return [dict(recipe=label,checkpoint=name,
                 correct_order=100*np.mean([r['adapted_rank']['correct_fraction'] for r in rows[1:]]),
                 uphill=100*np.mean([r['heldout_directional']['adapted_up_fraction'] for r in rows[1:]]))
            for name,rows in methods.items()]

legacy=old_cohort(report['legacy_methods'],'square')
radial_report=RADIAL_RUN/'comparison.json'
if radial_report.is_file():
    previous=json.loads(radial_report.read_text())
    new_config=json.loads((OUTPUT/'experiment.json').read_text())
    old_config=json.loads((RADIAL_RUN/'experiment.json').read_text())
    if (all(old_config[k]==new_config[k] for k in ('handcrafted','fitting','scale'))
            and old_config['manifest']['goal_ids']==new_config['manifest']['goal_ids']
            and all(previous[k]==report[k] for k in ('goal_ids','grid','heldout_mask'))):
        legacy=old_cohort(previous['methods'],'radial')+legacy
print('Original restricted cohort · mean per withheld goal (percent)')
display(pd.DataFrame(legacy).round(2))

## Full right-room heatmaps and collection coverage

Each panel uses its own unclipped cost scale. Set `HEATMAP_GOALS` to view other goals. Coverage plots show starts and retained observations for every stage.

In [ ]:
HEATMAP_GOALS=[0, 15]
grid=report['grid']
shape=(len(grid['ys']),len(grid['xs']))
plot_dir=OUTPUT/'heatmaps'
plot_dir.mkdir(exist_ok=True)
methods=list(report['methods'])

for goal_id in HEATMAP_GOALS:
    cols=5
    rows=(len(methods)+cols-1)//cols
    fig,axes=plt.subplots(rows,cols,figsize=(17,4.8*rows),constrained_layout=True)
    for ax in axes.flat[len(methods):]:
        ax.set_visible(False)
    for ax,method in zip(axes.flat,methods):
        row=report['methods'][method][goal_id]
        costs=np.asarray(row['adapted_costs'])[grid['point_ids']].reshape(shape)
        plot=ax.pcolormesh(grid['xs'],grid['ys'],costs,shading='nearest',cmap='viridis',
                           vmin=0,vmax=max(float(costs.max()),1e-9))
        ax.scatter(*row['goal'],marker='*',s=110,c='black',edgecolor='white')
        ax.set(title=method.replace('_',' ').title(),xlabel='Ball x',xlim=(35,60),
               ylim=(4,60),aspect='equal')
        fig.colorbar(plot,ax=ax,shrink=.6,label='Latent MSE')
    for ax in axes[:,0]:
        ax.set_ylabel('Ball y')
    fig.suptitle(f"Goal {goal_id} {tuple(row['goal'])} ({'withheld' if goal_id else 'seen'})")
    fig.savefig(plot_dir/f'goal_{goal_id}.png',dpi=130,bbox_inches='tight')
    plt.show()
    plt.close(fig)

In [ ]:
count=len(report['checkpoints'])
cols=4
rows=(count+cols-1)//cols
fig,axes=plt.subplots(rows,cols,figsize=(14,4.8*rows),sharey=True,constrained_layout=True)
for ax in axes.flat[count:]:
    ax.set_visible(False)
for ax,(band,stage) in zip(axes.flat,report['checkpoints'].items()):
    for row in stage['collection']:
        if row['retained_positions']:
            ax.scatter(*np.asarray(row['retained_positions']).T,s=6,c='tab:blue',alpha=.35)
        ax.scatter(*row['case']['start'],marker='x',s=22,c='tab:orange')
        ax.scatter(*row['case']['goal'],marker='*',s=50,c='black')
    ax.set(title=f"{band.replace('_',' ').title()} · {stage['retained_pairs']} pairs",
           xlim=(35,60),ylim=(4,60),aspect='equal',xlabel='Ball x')
for ax in axes[:,0]:
    ax.set_ylabel('Ball y')
fig.suptitle('Blue dots are retained observations, orange crosses are starts, black stars are collection goals')
fig.savefig(OUTPUT/'coverage.png',dpi=130,bbox_inches='tight')
plt.show()
plt.close(fig)